In [1]:
from jax import jit, grad

Current implementation of replace has implementational dependencies:

In [2]:
from flax.struct import PyTreeNode

class Rectangle(PyTreeNode):
    length: float
    width: float

class Square(PyTreeNode):
    side_length: float

    @property
    def length(self): return self.side_length

    @length.setter
    def length(self, value): self.side_length = value

@jit
def f(length):
    rect = Rectangle(4, 5)
    new_rect = rect.replace(length=length) # Replaces only length

    sq = Square(5)
    new_sq = sq.replace(length=length+1) # Replaces side_length

    return new_sq.length


grad(f)(6.)

%timeit f(6.)

TypeError: Square.__init__() got an unexpected keyword argument 'length'

Similarly when using simple_pytree with dataclass, even with explicitly defined getters and setters.

In [3]:
import simple_pytree

@simple_pytree.dataclass
class Rectangle(simple_pytree.Pytree):
    length: float
    width: float

@simple_pytree.dataclass
class Square(simple_pytree.Pytree):
    side_length: float

    @property
    def length(self): return self.side_length

    @length.setter
    def length(self, value): self.side_length = value


@jit
def f(length):
    rect = Rectangle(4, 5)
    new_rect = rect.replace(length=length) # Replaces only length

    sq = Square(5)
    new_sq = sq.replace(length=length+1) # Replaces side_length

    return new_sq.length


grad(f)(6.)

%timeit f(6.)

TypeError: Square.__init__() got an unexpected keyword argument 'length'

True even when we make the dataclass mutable:

In [4]:
@simple_pytree.dataclass
class Rectangle(simple_pytree.Pytree, mutable=True):
    length: float
    width: float

@simple_pytree.dataclass
class Square(simple_pytree.Pytree, mutable=True):
    side_length: float

    @property
    def length(self): return self.side_length

    @length.setter
    def length(self, value): self.side_length = value


@jit
def f(length):
    rect = Rectangle(4, 5)
    new_rect = rect.replace(length=length) # Replaces only length

    sq = Square(5)
    new_sq = sq.replace(length=length+1) # Replaces side_length

    return new_sq.length


grad(f)(6.)

%timeit f(6.)

TypeError: Square.__init__() got an unexpected keyword argument 'length'

We might hope that an externalized replace function that populates arguments to class.__init__ would work, but __init__ doesn't seem to follow getters and setters automatically:

In [5]:
from dataclasses import fields

@simple_pytree.dataclass
class Rectangle(simple_pytree.Pytree):
    length: float
    width: float

@simple_pytree.dataclass
class Square(simple_pytree.Pytree):
    side_length: float

    @property
    def length(self): return self.side_length
    
    @length.setter
    def length(self, value): self.side_length = value


def replace(instance, **kwargs):
    # Gather the current values for all attributes, replacing those specified in kwargs
    args = {
        field.name: kwargs.get(field.name, getattr(instance, field.name)) 
        for field in fields(instance)
        }

    # Create a new instance of the same class, passing the gathered arguments
    return instance.__class__(**args)


@jit
def f(length):
    rect = Rectangle(4, 5)
    new_rect = replace(rect, length=length) # Replaces only length

    sq = Square(5)
    new_sq = replace(sq, width=length+1) # Replaces side_length

    return new_sq.side_length

print(f(6))
grad(f)(6.)

%timeit f(6.)

5


TypeError: grad requires real-valued outputs (output dtype that is a sub-dtype of np.floating), but got int32. For differentiation of functions with integer outputs, use jax.vjp directly.

Instead, we use a mutable dataclass and perform replacing by performing __setattr__ on a copy.

In [9]:
from copy import copy

class Rectangle(simple_pytree.Pytree, mutable=True):

    def __init__(self, length, width):
        self.length = length
        self.width = width


class Square(simple_pytree.Pytree, mutable=True):

    def __init__(self, side_length): self.side_length = side_length

    @property
    def length(self): return self.side_length
    
    @length.setter
    def length(self, value): self.side_length = value

    @property
    def width(self): return self.side_length

    @width.setter
    def width(self, value): self.side_length = value


def replace(instance, **kwargs):
    new_instance = copy(instance)

    for attr, value in kwargs.items():
        setattr(new_instance, attr, value)

    return new_instance


@jit
def f(length):
    rect = Rectangle(4, 5)
    new_rect = replace(rect, length=length) # Replaces only length

    sq = Square(5)
    new_sq = replace(sq, width=length+1) # Replaces side_length

    return new_rect.length


grad(f)(6.)

%timeit f(6.)

447 µs ± 29 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [10]:
from copy import copy

@simple_pytree.dataclass
class Rectangle(simple_pytree.Pytree, mutable=True):
    length: float
    width: float

@simple_pytree.dataclass
class Square(simple_pytree.Pytree, mutable=True):

    # def __init__(self, side_length): self.side_length = side_length
    side_length: float

    @property
    def length(self): return self.side_length
    
    @length.setter
    def length(self, value): self.side_length = value

    @property
    def width(self): return self.side_length

    @width.setter
    def width(self, value): self.side_length = value


def replace(instance, **kwargs):
    new_instance = copy(instance)

    for attr, value in kwargs.items():
        setattr(new_instance, attr, value)

    return new_instance


@jit
def f(length):
    rect = Rectangle(4, 5)
    new_rect = replace(rect, length=length) # Replaces only length

    sq = Square(5)
    new_sq = replace(sq, width=length+1) # Replaces side_length

    return new_rect.length


grad(f)(6.)

%timeit f(6.)

436 µs ± 26.5 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


But then we could always just set attributes directly if the setters are already implemented..?

In [13]:
from copy import copy

@simple_pytree.dataclass
class Rectangle(simple_pytree.Pytree, mutable=True):
    length: float
    width: float

@simple_pytree.dataclass
class Square(simple_pytree.Pytree, mutable=True):

    side_length: float

    @property
    def length(self): return self.side_length
    
    @length.setter
    def length(self, value): self.side_length = value

    @property
    def width(self): return self.side_length

    @width.setter
    def width(self, value): self.side_length = value


def replace(instance, **kwargs):
    new_instance = copy(instance)

    for attr, value in kwargs.items():
        setattr(new_instance, attr, value)

    return new_instance


@jit
def f(length):
    new_rect = Rectangle(4, 5)
    new_rect.length=length # Replaces only length

    new_sq = Square(5)
    new_sq.length = length # Replaces only length

    return new_sq.length


grad(f)(6.)

%timeit f(6.)

434 µs ± 33.8 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


Alternatively, I could stick with immutable dataclasses and make the replace method work by instead providing implementations of init that follow the interface of the parent class. But this seems to have bad semantics. If I allow "Square" to init with a length and a width like "Rectangle", then I'd imply that Square can have distinct length and width values. But it can't. So I'd have to raise an exception if the user tries to init a Square with a length and a width. This seems like a bad idea. So instead, I'll just make sure the getters and setters for Square are consistent with the interface of Rectangle. Cool, geez.

Do jit functions refuse side effects when passed mutable objects?

In [25]:
@simple_pytree.dataclass
class Rectangle(simple_pytree.Pytree, mutable=True):
    length: float
    width: float

@jit
def grow_and_retrieve_area(rect: Rectangle, scale: float) -> float:
    rect.length = rect.length * scale
    rect.width = rect.width * scale
    return rect.length * rect.width

rect = Rectangle(3.0, 4.0)
print(grow_and_retrieve_area(rect, 1.5))
print(rect)

%timeit grow_and_retrieve_area(rect, 1.5)

27.0
Rectangle(length=3.0, width=4.0)
265 µs ± 18.4 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


Yes, but the problem is that the function behaves differently depending on whether the compilation actually happens or not!

In [17]:
@simple_pytree.dataclass
class Rectangle(simple_pytree.Pytree, mutable=True):
    length: float
    width: float

# @jit
def grow_and_retrieve_area(rect: Rectangle, scale: float) -> float:
    rect.length = rect.length * scale
    rect.width = rect.width * scale
    return rect.length * rect.width

rect = Rectangle(3.0, 4.0)
print(grow_and_retrieve_area(rect, 1.5))
print(rect)

27.0
Rectangle(length=4.5, width=6.0)


That's why I need replace -- or to make copies.

In [26]:
@simple_pytree.dataclass
class Rectangle(simple_pytree.Pytree, mutable=True):
    length: float
    width: float

@jit
def grow_and_retrieve_area(rect: Rectangle, scale: float) -> float:
    new_rect = copy(rect)
    new_rect.length = new_rect.length * scale
    new_rect.width = new_rect.width * scale
    return new_rect.length * new_rect.width

rect = Rectangle(3.0, 4.0)
print(grow_and_retrieve_area(rect, 1.5))
print(rect)

%timeit grow_and_retrieve_area(rect, 1.5)

27.0
Rectangle(length=3.0, width=4.0)
275 µs ± 23 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [27]:
def replace(instance, **kwargs):
    new_instance = copy(instance)

    for attr, value in kwargs.items():
        setattr(new_instance, attr, value)

    return new_instance

@simple_pytree.dataclass
class Rectangle(simple_pytree.Pytree, mutable=True):
    length: float
    width: float

@jit
def grow_and_retrieve_area(rect: Rectangle, scale: float) -> float:
    new_rect = replace(
        rect, 
        length = rect.length * scale, 
        width = rect.width * scale)
    return new_rect.length * new_rect.width

rect = Rectangle(3.0, 4.0)
print(grow_and_retrieve_area(rect, 1.5))
print(rect)

%timeit grow_and_retrieve_area(rect, 1.5)

27.0
Rectangle(length=3.0, width=4.0)
275 µs ± 26.5 µs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


It seems that the compiler will practically ignore copy. So okay.

In [1]:
def replace(instance, **kwargs):
    new_instance = copy(instance)

    for attr, value in kwargs.items():
        setattr(new_instance, attr, value)

    return new_instance

@simple_pytree.dataclass
class Rectangle(simple_pytree.Pytree, mutable=True):
    length: float
    width: float

@jit
def grow_and_retrieve_area(rect: Rectangle, scale: float) -> float:
    new_rect = replace(
        rect, 
        length = rect.length * scale, 
        width = rect.width * scale)
    return new_rect.length * new_rect.width

rect = Rectangle(3.0, 4.0)
print(grow_and_retrieve_area(rect, 1.5))
print(rect)

%timeit grow_and_retrieve_area(rect, 1.5)

NameError: name 'simple_pytree' is not defined